# Tutorial: Searching CSDA STAC catalog for multiple sites

This tutorial explains how to search the NASA CSDA SpatioTemporal Asset Catalog (STAC) for imagery covering one or more evaluation sites. It is adapted from `csda_stac_search_multiple.ipynb` and intentionally stops after the search results and their footprints have been written to disk.

The workflow has five conceptual stages:

1. Read geographic areas of interest (AOIs) for the sites.
2. Convert those AOIs into search geometries.
3. Connect to the STAC API and choose collections and dates.
4. Search the catalog, potentially using several processes.
5. Combine the results into a GeoDataFrame and write them out.

A STAC catalog is a standardized index of geospatial assets. Each catalog item usually describes one acquisition, including its footprint, date, collection, and links to files or other assets. A STAC search returns metadata about items; it does not automatically download the imagery.  

`Paul Montesano, PhD | NASA Goddard Space Flight Center | 2026`

## Before you begin

This notebook assumes that your Python environment already contains the project libraries used by the original workflow, including `geopandas`, `pandas`, `numpy`, `matplotlib`, `pystac-client`, `tqdm`, and the local modules `staclib`, `csdalib`, and `siteslib` available when you Git clone the repository `csda_summaries`.

`https://github.com/pahbs/csda_summaries/`

**This is a PUBLIC repo with prototype code and notebooks that I offer as a convenience.**

The local module paths below are examples from the original notebook. Replace them with the paths used by your own environment.

In [1]:
import sys
from datetime import datetime
from functools import partial
from multiprocessing import Pool, cpu_count
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Change to your 'csda_summaries' repository location
sys.path.append('/home/pmontesa/code/csda_summaries/lib')

import csdalib
import siteslib
import staclib

In [34]:
from pathlib import Path
STAC_CACHE_PATH = Path("/explore/nobackup/people/pmontesa/stac_cache")
STAC_CACHE_PATH.mkdir(parents=True, exist_ok=True)

## 1. Read the evaluation-site AOIs

An area of interest, or AOI, is the geographic area used in a search. Here the AOIs are stored as a GeoJSON file. A GeoDataFrame combines a tabular structure with a geometry column, allowing site attributes and spatial operations to be handled together.

The original workflow reads the evaluation-site file directly from GitHub. Reading a version-controlled source makes the analysis easier to reproduce because the source URL identifies the input dataset.

In [36]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'

SITES_LABEL = 'eval' # 'eval' or 'csda'
SITES_FN = f'{SITES_LABEL}_sites_aoi.geojson'
SITES_URL = f'{RAW_BASE}/sites/{SITES_FN}'

sites_gdf = gpd.read_file(SITES_URL)
print(f'{len(sites_gdf)} site AOIs were read.')
print('Available fields:')
print(list(sites_gdf.columns))
display(sites_gdf.head())

617 site AOIs were read.
Available fields:
['site_id', 'Site Name abbrev', 'Site Name', 'Location Name', 'Country', 'Program Use', 'Longitude', 'Latitude', 'Remote Sensing Domain', 'Priority Level', 'Evaluation Category', 'Source', 'Surface Domain', 'Assessment type(s)', 'Resolution Category', 'aoi_shape', 'aoi_size_km', 'max_view_angle', 'min_num_acqs', 'ideal_num_acqs', 'max_cloud_pct_aoi', 'max_cloud_pct_scene', 'contact', 'reference', 'notes', 'area_km2', 'geometry']


,site_id,Site Name abbrev,Site Name,Location Name,Country,Program Use,Longitude,Latitude,Remote Sensing Domain,Priority Level,...,max_view_angle,min_num_acqs,ideal_num_acqs,max_cloud_pct_aoi,max_cloud_pct_scene,contact,reference,notes,area_km2,geometry
0,NaN,None,Baoshan,Baoshan,China,CSDA,99.1672,25.0547,Optical Multi/Hyper,high,...,30.0,5.0,5.0,0.0,15.0,None,None,None,9.000000,"POLYGON ((99.18205 25.04114, 99.18209 25.06823..."
1,NaN,None,Cairns,Cairns,Australia,CSDA,145.7649,-16.9260,Optical Multi/Hyper,high,...,30.0,5.0,5.0,0.0,15.0,None,None,None,9.000000,"POLYGON ((145.77890 -16.93964, 145.77907 -16.9..."
2,NaN,None,Copiapo,Copiapo,Chile,CSDA,-70.3189,-27.3681,Optical Multi/Hyper,high,...,30.0,5.0,5.0,0.0,15.0,None,None,None,9.000000,"POLYGON ((-70.30389 -27.38178, -70.30358 -27.3..."
3,NaN,None,Luanda,Luanda,Angola,CSDA,13.2315,-8.8541,Optical Multi/Hyper,high,...,30.0,5.0,5.0,0.0,15.0,None,None,None,9.000000,"POLYGON ((13.24507 -8.86773, 13.24520 -8.84060..."
4,NaN,None,Manta,Manta,Ecuador,CSDA,-80.7115,-0.9642,Optical Multi/Hyper,high,...,30.0,5.0,5.0,0.0,15.0,None,None,None,8.993031,"POLYGON ((-80.69802 -0.97777, -80.69802 -0.950..."


### Inspect the site table before searching

Inspection is a basic quality-control step. Confirm that the site-name field exists, that the geometry column is populated, and that the coordinate reference system is defined. The coordinate reference system (CRS) describes how coordinates relate to locations on Earth.

In [37]:
SITE_NAME_FIELD = 'Site Name'

if SITE_NAME_FIELD not in sites_gdf.columns:
    raise KeyError(f'Missing {SITE_NAME_FIELD!r}; available fields: {list(sites_gdf.columns)}')
if sites_gdf.geometry.isna().any():
    print('At least one site has missing geometry.')

print(f'CRS: {sites_gdf.crs}')
print(f'Unique site names: {sites_gdf[SITE_NAME_FIELD].nunique()}')
display(sites_gdf[[SITE_NAME_FIELD, 'geometry']].head())

CRS: epsg:4326
Unique site names: 616


,Site Name,geometry
0,Baoshan,"POLYGON ((99.18205 25.04114, 99.18209 25.06823..."
1,Cairns,"POLYGON ((145.77890 -16.93964, 145.77907 -16.9..."
2,Copiapo,"POLYGON ((-70.30389 -27.38178, -70.30358 -27.3..."
3,Luanda,"POLYGON ((13.24507 -8.86773, 13.24520 -8.84060..."
4,Manta,"POLYGON ((-80.69802 -0.97777, -80.69802 -0.950..."


## 2. Choose the sites and prepare map geometries

The original notebook supports several search modes. You can search all evaluation sites, select one site, or use a separate site file. The variables below make that choice explicit.

A buffer expands a geometry outward by a specified distance. The search buffer and the display buffer are kept separate because a map may need more surrounding context than the actual STAC query.

In [38]:
DO_SITE_SUBSET = True

In [39]:
SITE_NAME = 'Railroad Valley'
BUFFER_KM_FOR_SEARCH = 0
BUFFER_KM_FOR_DISPLAY = 0

if DO_SITE_SUBSET:

    SITENAME = 'SLI_TEST'
    SITES_STR = SITENAME.replace(' ','.')
    SITE_LIST = ['Railroad Valley','Old Bahia Bridge','belmanip2-078','MD_Science_Center']
    
    sites_gdf = sites_gdf[sites_gdf[SITE_NAME_FIELD].isin(SITE_LIST)]
    out_fn = STAC_CACHE_PATH / SITES_FN.replace('.geojson',f'_{SITENAME}.geojson')
    sites_gdf.to_file(out_fn)
    print(str(out_fn))

    # sites_gdf = sites_gdf.loc[sites_gdf[SITE_NAME_FIELD] == SITE_NAME].copy()
    # if sites_gdf.empty:
    #     raise ValueError(f'No AOI found for site {SITE_NAME!r}.')
    SITES_LABEL = SITE_NAME.replace(' ', '.')

if sites_gdf.crs is None:
    raise ValueError('The site GeoDataFrame has no CRS.')

# Web Mercator uses metres, so it is convenient for a simple map buffer.
sites_gdf_buffered = sites_gdf.to_crs(3857).copy()
sites_gdf_buffered['geometry'] = sites_gdf_buffered.geometry.buffer(BUFFER_KM_FOR_SEARCH * 1000)
sites_gdf_buffered = sites_gdf_buffered.to_crs(4326)
sites_gdf_buffered = sites_gdf_buffered.loc[~sites_gdf_buffered.geometry.is_empty].copy()

sites_gdf_display = sites_gdf.to_crs(3857).copy()
sites_gdf_display['geometry'] = sites_gdf_display.geometry.buffer(BUFFER_KM_FOR_DISPLAY * 1000)
sites_gdf_display = sites_gdf_display.to_crs(4326)
display(sites_gdf_display[[SITE_NAME_FIELD, 'geometry']])

/explore/nobackup/people/pmontesa/stac_cache/eval_sites_aoi_SLI_TEST.geojson


,Site Name,geometry
40,Old Bahia Bridge,"POLYGON ((-81.28465 24.65069, -81.29453 24.650..."
53,Railroad Valley,"POLYGON ((-115.65561 38.49661, -115.65582 38.4..."
378,belmanip2-078,"POLYGON ((-123.65343 42.60278, -123.65358 42.6..."
479,MD_Science_Center,"POLYGON ((-76.58223 39.28348, -76.58234 39.280..."


## 3. Create search regions

Many sites can produce a large number of STAC requests. The helper function `create_global_grid` divides the globe into rectangular cells. Searching by cells reduces the number of very large requests and makes a global search practical.

For a site-only search, the project helper `create_site_dict_from_gdf` converts each AOI into the dictionary format expected by `stac_search_site`. Each dictionary entry contains a name and the geometry information needed by the search function.

In [40]:
DO_GLOBAL_SEARCH = False

In [41]:
def create_global_grid(grid_size_degrees=30):
    """Create rectangular longitude-latitude search cells."""
    site_dicts = {}
    lons = range(-180, 180, grid_size_degrees)
    lats = range(-90, 90, grid_size_degrees)

    for i, lon_start in enumerate(lons):
        for j, lat_start in enumerate(lats):
            lon_end = min(lon_start + grid_size_degrees, 180)
            lat_end = min(lat_start + grid_size_degrees, 90)
            grid_name = f'grid_{i:02d}_{j:02d}'
            site_dicts[grid_name] = {
                'bbox': [lon_start, lat_start, lon_end, lat_end],
                'lon': (lon_start + lon_end) / 2,
                'lat': (lat_start + lat_end) / 2,
                'buffer': None,
                'geom_type': 'Polygon',
            }
    return site_dicts

if DO_GLOBAL_SEARCH:
    site_dicts = create_global_grid(grid_size_degrees=10)
    print(f'Global search divided into {len(site_dicts)} cells.')
else:
    site_dicts = staclib.create_site_dict_from_gdf(
        sites_gdf_buffered,
        site_name_field=SITE_NAME_FIELD,
        buffer_degrees=0,
    )
    print(f'Site search contains {len(site_dicts)} AOIs.')

CSDA_SEARCH_REGIONS = list(site_dicts.items())
display(CSDA_SEARCH_REGIONS[:2])

Site search contains 4 AOIs.


[('Old Bahia Bridge',
  {'bbox': [-81.29455175716544,
    24.650674701517772,
    -81.28464859834581,
    24.659725137258015],
   'lon': -81.28959999953068,
   'lat': 24.655199971568393,
   'buffer': None,
   'geom_type': 'Polygon'}),
 ('Railroad Valley',
  {'bbox': [-115.72439439273283,
    38.46997061448379,
    -115.65560626354042,
    38.524029255167235],
   'lon': -115.69000014472898,
   'lat': 38.4970012053502,
   'buffer': None,
   'geom_type': 'Polygon'})]

## 4. Connect to the CSDA STAC API

The STAC endpoint is a web API. Opening the catalog tests that the endpoint can be reached and that it responds as a STAC catalog. The small test search is useful because a successful network connection does not guarantee that search requests are configured correctly.

In [42]:
import pystac_client

STAC_URL = 'https://csdap.earthdata.nasa.gov/stac'
catalog = pystac_client.Client.open(STAC_URL)
print(f'Connected to catalog: {catalog.id}')

test_items = list(catalog.search(max_items=1).items())
print(f'Test search returned {len(test_items)} item(s).')

Connected to catalog: stac-fastapi
Test search returned 1 item(s).


## 5. Define the search query

A STAC query is defined by several filters:

- `collections` limits the search to selected data providers or products.
- `start_date` and `end_date` define the time interval. The end date is written as the start of the following interval in this example.
- `months` provides an additional month filter through the project helper.

The exact collection IDs must be checked against the catalog. A collection name that is visually similar but not identical will return no results or an API error.

In [43]:
TODAY = datetime.now().strftime('%Y-%m-%d')
START_OF_COMMERCIAL_ERA = '1999-01-01'

#COLLECTION_LIST = ['planet', 'airbus-optical', 'satellogic', 'maxar-legion']
COLLECTION_LIST = ['airbus-optical']

YEAR = 2020
START_DATE = START_OF_COMMERCIAL_ERA #f'{YEAR}-01-01'
END_DATE = TODAY #f'{YEAR + 1}-01-01'
MONTH_LIST = [6, 7, 8]

print('Collections:', COLLECTION_LIST)
print('Date interval:', START_DATE, 'through', END_DATE)
print('Months:', MONTH_LIST)

Collections: ['airbus-optical']
Date interval: 1999-01-01 through 2026-10-02
Months: [6, 7, 8]


## 6. Search the catalog for every region

`staclib.stac_search_site` performs the project-specific search and returns a tuple containing a site-level GeoDataFrame, STAC items, and a status message.

The `partial` function fixes the query parameters while leaving the search region as the argument that changes on each call. The pool then distributes those independent searches across worker processes. Start with one process when debugging; parallel requests are faster for large searches but can place more load on the remote service.

In [44]:
search_func = partial(
    staclib.stac_search_site,
    end_date=END_DATE,
    start_date=START_DATE,
    months=MONTH_LIST,
    collections=COLLECTION_LIST,
)

USE_MULTIPROCESSING = True
n_processes = max(1, cpu_count() - 2) if USE_MULTIPROCESSING else 1
print(f'Using {n_processes} process(es) for {len(CSDA_SEARCH_REGIONS)} search regions.')

if n_processes == 1:
    results = [search_func(region) for region in tqdm(CSDA_SEARCH_REGIONS)]
else:
    with Pool(processes=n_processes) as pool:
        results = list(tqdm(pool.imap(search_func, CSDA_SEARCH_REGIONS), total=len(CSDA_SEARCH_REGIONS)))

stac_gdf_list = []
stac_items_list = []
print('Results by search region:')
for region_name, gdf, items, message in results:
    print(f'  {region_name}: {message}')
    if gdf is not None and not gdf.empty:
        stac_gdf_list.append(gdf)
        stac_items_list.extend(items)

if stac_gdf_list:
    stac_gdf = gpd.GeoDataFrame(pd.concat(stac_gdf_list, ignore_index=True))
    print(f'Total: {len(stac_gdf)} items.')
    if 'site_name' in stac_gdf.columns:
        print(f"Sites represented: {stac_gdf['site_name'].nunique()}")
        display(stac_gdf['site_name'].value_counts())
else:
    stac_gdf = gpd.GeoDataFrame()
    print('No STAC items were found.')

Using 38 process(es) for 4 search regions.


  0%|          | 0/4 [00:00<?, ?it/s]

Results by search region:
  Old Bahia Bridge: No items found
  Railroad Valley: Found 8 items across 28 interval(s) (months=[6, 7, 8])
  belmanip2-078: No items found
  MD_Science_Center: No items found
Total: 8 items.
Sites represented: 1


Railroad Valley    8
Name: site_name, dtype: int64

## 7. Write the STAC items and footprint table

The search produces two useful representations of the result:

- `stac_items_list` contains STAC item objects and preserves the catalog metadata structure.
- `stac_gdf` is a tabular spatial representation in which each row is an item and the geometry is its footprint.

Asset fields can contain lists or nested objects that are inconvenient in a GeoPackage table. The final cleanup removes columns whose names contain `asset`, converts remaining list-valued fields to strings, and writes the result as a GeoPackage. The original STAC item objects are also written to JSON.

In [45]:
stac_items_list[0]

id: PHR1A_202508311842183_P_ORT_RFL
"bbox: [-115.800614218728, 38.40889354455381, -115.5795937064113, 38.5850024828641]"
gsd: 0.5
datetime: 2025-08-31T18:42:18.300000Z
platform: PHR1A
proj:code: EPSG:32611
"proj:shape: [38553, 38009]"
view:azimuth: 180.0172297674612
constellation: PHR
csda:item_type: PHR
eo:cloud_cover: 2.7


In [46]:
from pystac import ItemCollection

if not stac_gdf.empty:
    collection_label = '_'.join(COLLECTION_LIST)
    output_stac_json = Path(f'stac_items_{SITES_LABEL}_{collection_label}_{YEAR}.json')
    output_footprints = Path(f'stac_{SITES_LABEL}_{collection_label}_{YEAR}.gpkg')

    ItemCollection(stac_items_list).save_object(dest_href=str(output_stac_json))

    stac_gdf_to_write = stac_gdf.drop(
        columns=[column for column in stac_gdf.columns if 'asset' in column.lower()],
        errors='ignore',
    ).copy()
    for column in stac_gdf_to_write.columns:
        if stac_gdf_to_write[column].map(lambda value: isinstance(value, list)).any():
            stac_gdf_to_write[column] = stac_gdf_to_write[column].map(
                lambda value: str(value) if isinstance(value, list) else value
            )

    stac_gdf_to_write.to_file(output_footprints, driver='GPKG')
    print(f'Wrote STAC items to {output_stac_json}')
    print(f'Wrote footprint GeoPackage to {output_footprints}')
    out_csv_fn = str(output_footprints).replace('.gpkg','.csv')
    stac_gdf_to_write.to_csv(out_csv_fn, index=False)
    print(f'Wrote CSV to {out_csv_fn}')
else:
    print('Nothing was written because stac_gdf is empty.')

Wrote STAC items to stac_items_Railroad.Valley_airbus-optical_2020.json
Wrote footprint GeoPackage to stac_Railroad.Valley_airbus-optical_2020.gpkg
Wrote CSV to stac_Railroad.Valley_airbus-optical_2020.csv


## What has been produced?

At this point the workflow has completed the catalog-search stage. The JSON file preserves the STAC item records, while the GeoPackage provides a GIS-friendly footprint table for inspection in software such as QGIS. Neither file contains downloaded image pixels.

Before continuing to image download or scientific analysis, inspect the number of returned items, their acquisition dates, collection names, spatial footprints, and any duplicate acquisitions. Those checks determine whether the search query represents the intended population of observations.

In [48]:
stac_gdf_to_write

,id,collection,datetime,geometry,bbox,gsd,platform,proj:code,proj:shape,view:azimuth,...,view:off_nadir,processing:level,view:sun_azimuth,view:sun_elevation,view:incidence_angle,airbus-optical:imaging_mode,airbus-optical:spectral_processing,airbus-optical:radiometric_processing,eo:snow_cover,site_name
0,PHR1A_202508311842183_P_ORT_RFL,airbus-optical,2025-08-31T18:42:18.300000Z,"POLYGON ((-115.79773 38.58500, -115.57959 38.5...","[-115.800614218728, 38.40889354455381, -115.57...",0.5,PHR1A,EPSG:32611,"[38553, 38009]",180.017230,...,3.746460,ORT,151.467885,57.038532,4.047683,Mono,P,REFLECTANCE,NaN,Railroad Valley
1,PHR1A_202508311842183_MS_ORT_RFL,airbus-optical,2025-08-31T18:42:18.300000Z,"POLYGON ((-115.79774 38.58500, -115.57960 38.5...","[-115.8006227702064, 38.40889590077379, -115.5...",2.0,PHR1A,EPSG:32611,"[9639, 9503]",180.017230,...,3.746460,ORT,151.467885,57.038532,4.047683,Mono,MS,REFLECTANCE,NaN,Railroad Valley
2,PHR1A_202508241846120_P_ORT_RFL,airbus-optical,2025-08-24T18:46:12Z,"POLYGON ((-115.79773 38.58500, -115.57959 38.5...","[-115.800614218728, 38.40889354455381, -115.57...",0.5,PHR1A,EPSG:32611,"[38553, 38009]",179.947446,...,10.089028,ORT,150.485681,59.506037,11.196493,Mono,P,REFLECTANCE,NaN,Railroad Valley
3,PHR1A_202508241846120_MS_ORT_RFL,airbus-optical,2025-08-24T18:46:12Z,"POLYGON ((-115.79774 38.58500, -115.57960 38.5...","[-115.8006227702064, 38.40889590077379, -115.5...",2.0,PHR1A,EPSG:32611,"[9639, 9503]",179.947446,...,10.089028,ORT,150.485681,59.506037,11.196493,Mono,MS,REFLECTANCE,NaN,Railroad Valley
4,PNEO4_202508241840402_PAN_ORT_RFL,airbus-optical,2025-08-24T18:40:40.200000Z,"POLYGON ((-115.76484 38.55726, -115.61332 38.5...","[-115.76686818818027, 38.43663842413944, -115....",0.3,PNEO4,EPSG:32611,"[44000, 44020]",292.926914,...,5.516528,ORT,147.721838,59.237688,5.992504,Mono,PAN,REFLECTANCE,0.0,Railroad Valley
5,PNEO4_202508241840402_MS_ORT_RFL,airbus-optical,2025-08-24T18:40:40.200000Z,"POLYGON ((-115.76484 38.55726, -115.61333 38.5...","[-115.76686805021122, 38.43664665459069, -115....",1.2,PNEO4,EPSG:32611,"[11000, 11005]",292.926914,...,5.516528,ORT,147.721838,59.237688,5.992504,Mono,MS,REFLECTANCE,0.0,Railroad Valley
6,PHR1B_202508181841594_P_ORT_RFL,airbus-optical,2025-08-18T18:41:59.400000Z,"POLYGON ((-115.79773 38.58500, -115.57959 38.5...","[-115.800614218728, 38.40889354455381, -115.57...",0.5,PHR1B,EPSG:32611,"[38553, 38009]",179.996176,...,5.360457,ORT,146.332231,60.765919,5.787773,Mono,P,REFLECTANCE,NaN,Railroad Valley
7,PHR1B_202508181841594_MS_ORT_RFL,airbus-optical,2025-08-18T18:41:59.400000Z,"POLYGON ((-115.79774 38.58500, -115.57960 38.5...","[-115.8006227702064, 38.40889590077379, -115.5...",2.0,PHR1B,EPSG:32611,"[9639, 9503]",179.996176,...,5.360457,ORT,146.332231,60.765919,5.787773,Mono,MS,REFLECTANCE,NaN,Railroad Valley


In [47]:
stac_gdf_to_write.iloc[0]

id                                                         PHR1A_202508311842183_P_ORT_RFL
collection                                                                  airbus-optical
datetime                                                       2025-08-31T18:42:18.300000Z
geometry                                 POLYGON ((-115.7977304602179 38.5850024828641,...
bbox                                     [-115.800614218728, 38.40889354455381, -115.57...
gsd                                                                                    0.5
platform                                                                             PHR1A
proj:code                                                                       EPSG:32611
proj:shape                                                                  [38553, 38009]
view:azimuth                                                                     180.01723
constellation                                                                          PHR